# Part 4: Graph Modularity and Community Detection

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

## Question 1

### Question (b): Build the 6-node adjacency matrix

In [ ]:
A = np.zeros((6, 6), dtype=float)

# node 0 (R node 1)
A[0, 1] = 1
A[0, 2] = 1
# node 1 (R node 2)
A[1, 0] = 1
A[1, 2] = 1
# node 2 (R node 3)
A[2, 0] = 1
A[2, 1] = 1
A[2, 3] = 1
# node 3 (R node 4)
A[3, 2] = 1
A[3, 4] = 1
A[3, 5] = 1
# node 4 (R node 5)
A[4, 3] = 1
A[4, 5] = 1
# node 5 (R node 6)
A[5, 3] = 1
A[5, 4] = 1

print("Adjacency matrix A:")
print(A)

### Question (b): Build and plot the graph

In [ ]:
G = nx.from_numpy_array(A, create_using=nx.DiGraph)

fig, ax = plt.subplots(figsize=(5, 4))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, with_labels=True, node_color='lightblue',
        node_size=600, arrows=True, arrowsize=15)
ax.set_title('6-node directed graph')
plt.tight_layout()
plt.show()

### Question (b): Compute the modularity matrix B

In [ ]:
D_deg = A.sum(axis=1)          # degree vector
m     = A.sum() / 2            # total number of (undirected) edges
B     = A - np.outer(D_deg, D_deg) / (2 * m)

print("Modularity matrix B:")
print(np.round(B, 4))

### Question (b): Leading eigenvector and community membership

In [ ]:
w, v = np.linalg.eig(B)

# Sort eigenvalues descending and take the leading eigenvector
sort_idx       = np.argsort(w)[::-1]
w_sorted       = w[sort_idx].real
v_sorted       = v[:, sort_idx].real
leading_vector = v_sorted[:, 0]

print("Eigenvalues (descending):", np.round(w_sorted, 4))
print("Leading eigenvector:      ", np.round(leading_vector, 4))

In [ ]:
# Assign membership by sign of leading eigenvector
# Negative -> community 1, Positive -> community 2  (matching R convention)
membership = np.where(leading_vector < 0, 1, 2)
print("Community membership (1-indexed nodes):")
for node, comm in enumerate(membership):
    print(f"  Node {node + 1}: community {comm}")

### Question (b): Compute modularity Q manually

In [ ]:
# Q = (1 / 2m) * sum_{ij} B_ij * delta(c_i, c_j)
Q = 0.0
for i in range(6):
    for j in range(6):
        if membership[i] == membership[j]:
            Q += B[i, j]
Q /= (2 * m)
print(f"Modularity Q = {Q:.4f}")

### Question (c): Interpretation

The magnitude of each coordinate in the leading eigenvector reflects the degree of certainty when assigning a node to a community. Nodes 3 and 4 (0-indexed: nodes 2 and 3) have smaller magnitudes compared to the others, which makes sense because they sit on the frontier between the two communities — node 3 bridges the two dense triangles.

### Question (d): Splitting into more than two communities

To detect more than two communities, the spectral algorithm can be applied recursively: recompute a new modularity sub-matrix for each detected community and check whether its leading eigenvalue is positive. If so, split the sub-graph further.

### Question (e): All-negative eigenvalues

When all eigenvalues of the modularity matrix are negative, the maximum-modularity split corresponds to the zero eigenvalue (associated with the constant all-ones eigenvector). This means no partition improves on the null model and the graph should not be divided into communities.

---

## Question 2

### Question (a): Build the 5-node graph and compute cosine similarity

In [ ]:
A2 = np.zeros((5, 5), dtype=float)

# node 0 (R node 1)
A2[0, 1] = 1
A2[0, 3] = 1
# node 1 (R node 2)
A2[1, 0] = 1
A2[1, 2] = 1
A2[1, 3] = 1
# node 2 (R node 3)
A2[2, 1] = 1
A2[2, 3] = 1
A2[2, 4] = 1
# node 3 (R node 4)
A2[3, 0] = 1
A2[3, 1] = 1
A2[3, 2] = 1
A2[3, 4] = 1
# node 4 (R node 5)
A2[4, 2] = 1
A2[4, 3] = 1

print("Adjacency matrix A2:")
print(A2)

In [ ]:
# Cosine similarity matrix S
S = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        num   = A2[i] @ A2[j]
        denom = np.sqrt(A2[i] @ A2[i]) * np.sqrt(A2[j] @ A2[j])
        S[i, j] = num / denom if denom > 0 else 0.0

print("Cosine similarity matrix S:")
print(np.round(S, 4))

### Question (b): Hierarchical clustering and dendrogram

`hclust` in R requires a *distance* matrix. We use $\mathbf{D} = 1 - \mathbf{S}$ and single linkage.

In [ ]:
D_dist = 1.0 - S

# scipy linkage expects a condensed distance vector (upper triangle)
# squareform converts the full symmetric matrix to condensed form
condensed = squareform(D_dist, checks=False)

Z = linkage(condensed, method='single')

fig, ax = plt.subplots(figsize=(6, 4))
dendrogram(Z, labels=[f'Node {i+1}' for i in range(5)], ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram (single linkage)')
ax.set_xlabel('Node')
ax.set_ylabel('Distance (1 - cosine similarity)')
plt.tight_layout()
plt.show()